## Investigating genetic similarity labels ("ANCESTRY_CALL")

In this notebook we take the provided ancestry calls (determined by an NNLS hard-calling approach developed by Genomics Ltd) and look deeper into them

In [ ]:
## Load in the ancestry call data

system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download imputed_resources/ofh_imputed_sample_ancestry.v6.tsv'")
genetic_similarity <- read.table("ofh_imputed_sample_ancestry.v6.tsv", header = TRUE, sep = "\t")

In [ ]:
## Counts of each ancestry call, displayed in descending order

library(dplyr)

genetic_similarity %>%
    group_by(ANCESTRY_CALL) %>%
    summarise(n=n(), .groups = "drop") %>%
    arrange(desc(n))


print(paste("number of people WITH an assigned ancestry call: ", nrow(genetic_similarity%>%dplyr::filter(!is.na(ANCESTRY_CALL)))))
                                                                                                
print(paste("number of people without an assigned ancestry call: ", nrow(genetic_similarity%>%dplyr::filter(is.na(ANCESTRY_CALL)))))

In [ ]:
## Try to reconstruct the ancestry calls, then group as superpopulations (EUR, AFR, EAS, SAS, AMR)

region_cols <- c("C_S_UK", "N_ENG_S_SCOT", "NI_N_SCOT", "CW_EUROPE", 
                "IRELAND", "SE_EUROPE", "E_EUROPE", "INDIA_PAKISTAN", 
                "CE_ASIA", "SRI_LANKA", "PORTUGAL_SPAIN", "W_AFRICA", 
                "SW_WALES", "NW_WALES", "C_ASIA", "M_EAST_W_ASIA", 
                "BANGLADESH", "N_EUROPE", "N_AFRICA", "SE_ASIA", 
                "C_S_AFRICA", "N_C_S_AMERICA", "JAPAN_KOREA", 
                "E_AFRICA", "FINLAND")

genetic_similarity_reconstructed <- genetic_similarity %>%
    dplyr::mutate(
        ANCESTRY_CALL_2 = region_cols[
            max.col(pick(all_of(region_cols)), ties.method = "first")
        ]
    )



## Add superpopulation labels 

genetic_similarity_reconstructed <- genetic_similarity_reconstructed %>%
    dplyr::rename("pid" = "id") %>%
    dplyr::mutate(genetic_similarity_group = case_when(
        ANCESTRY_CALL %in% c("CW_EUROPE", "C_S_UK", "E_EUROPE", "FINLAND", "IRELAND", "NI_N_SCOT",
                             "NW_WALES", "N_ENG_S_SCOT", "N_EUROPE", "PORTUGAL_SPAIN", "SE_EUROPE",
                             "SW_WALES") ~ "EUR",
        ANCESTRY_CALL %in% c("C_S_AFRICA", "E_AFRICA", "N_AFRICA", "W_AFRICA") ~ "AFR",
        ANCESTRY_CALL %in% c("BANGLADESH", "INDIA_PAKISTAN", "SRI_LANKA") ~ "SAS",
        ANCESTRY_CALL %in% c("CE_ASIA", "SE_ASIA", "JAPAN_KOREA") ~ "EAS",
        ANCESTRY_CALL %in% c("C_ASIA") ~ "CAS",
        ANCESTRY_CALL %in% c("N_C_S_AMERICA") ~ "AMR",
        ANCESTRY_CALL %in% c("M_EAST_W_ASIA") ~ "MID",
        TRUE ~ "NA_character_"
    ), genetic_similarity_group_2 = case_when(
        ANCESTRY_CALL_2 %in% c("CW_EUROPE", "C_S_UK", "E_EUROPE", "FINLAND", "IRELAND", "NI_N_SCOT",
                               "NW_WALES", "N_ENG_S_SCOT", "N_EUROPE", "PORTUGAL_SPAIN", "SE_EUROPE",
                               "SW_WALES") ~ "EUR",
        ANCESTRY_CALL_2 %in% c("C_S_AFRICA", "E_AFRICA", "N_AFRICA", "W_AFRICA") ~ "AFR",
        ANCESTRY_CALL_2 %in% c("BANGLADESH", "INDIA_PAKISTAN", "SRI_LANKA") ~ "SAS",
        ANCESTRY_CALL_2 %in% c("CE_ASIA", "SE_ASIA", "JAPAN_KOREA") ~ "EAS",
        ANCESTRY_CALL_2 %in% c("C_ASIA") ~ "CAS",
        ANCESTRY_CALL_2 %in% c("N_C_S_AMERICA") ~ "AMR",
        ANCESTRY_CALL_2 %in% c("M_EAST_W_ASIA") ~ "MID",
        TRUE ~ "NA_character_"
    )
    )

##

write.csv(genetic_similarity_reconstructed, "genetic_similarity_reconstructed.csv", row.names = FALSE)

In [ ]:
## Check how often the reconstructed ancestry calls match the original ancestry calls

genetic_similarity_reconstructed %>%
    dplyr::filter(
        ANCESTRY_CALL != ANCESTRY_CALL_2
    ) %>% summarise(n = n())

genetic_similarity_reconstructed %>%
    dplyr::filter(
        genetic_similarity_group != genetic_similarity_group_2
    )

In [ ]:
## Let's take a closer look at those whose superpopulations do not match between the original and reconstructed calls

mismatched_participants <- genetic_similarity_reconstructed %>%
    dplyr::filter(genetic_similarity_group != genetic_similarity_group_2)

write.csv(mismatched_participants, "mismatched_participants.csv", row.names = FALSE)

df_for_table <- mismatched_participants %>%
    dplyr::mutate(genetic_similarity_group_2 = case_when(
        genetic_similarity_group_2 == "EUR" ~ "EUR_2",
        genetic_similarity_group_2 == "AFR" ~ "AFR_2",
        genetic_similarity_group_2 == "SAS" ~ "SAS_2",
        genetic_similarity_group_2 == "EAS" ~ "EAS_2",
        genetic_similarity_group_2 == "CAS" ~ "CAS_2",
        genetic_similarity_group_2 == "AMR" ~ "AMR_2",
        genetic_similarity_group_2 == "MID" ~ "MID_2",
        TRUE ~ genetic_similarity_group_2
    ))

table(df_for_table$genetic_similarity_group, df_for_table$genetic_similarity_group_2)